# Projeto Semestral I — Ciência de Dados
## CESOP 04832 — Percepção dos Brasileiros Acerca da Democracia

**Objetivo deste notebook:** carregar, estruturar, limpar e preparar as duas bases do
projeto (CESOP e TSE) para as análises do notebook seguinte. Aqui **não há análise
exploratória** — apenas o *pipeline* de preparação dos dados.

Ao final, são gerados em `data/processed/`:

| Arquivo | Conteúdo |
|---|---|
| `cesop_clean.parquet` | Base CESOP tratada (respondentes, perguntas recodificadas, variáveis derivadas) |
| `tse_clean.parquet` | TSE em linhas-detalhe, com taxas calculadas |
| `tse_uf.parquet` | TSE agregado por UF |
| `tse_perfil.parquet` | TSE agregado por perfil demográfico nacional |
| `cesop_labels.json` | Dicionário de rótulos das colunas do CESOP (referência) |

## 0. Configuração do ambiente

Centralizamos aqui imports e caminhos. **Por quê:** evita reescrever paths longos ao
longo do notebook e garante portabilidade. O bloco detecta automaticamente se está
rodando no **Google Colab** (requisito da entrega: *100% executável no Colab*) e, nesse
caso, monta o Google Drive; caso contrário, assume execução local a partir de
`notebooks/`. Assim o mesmo notebook roda nos dois ambientes sem edição manual.

In [ ]:
# Imports e definição de caminhos (com detecção de ambiente Colab x local).
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

# Detecta o ambiente: no Colab montamos o Drive; localmente usamos a raiz do repo.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Ajuste este caminho para a pasta do projeto no seu Drive, se necessário.
    PROJECT_ROOT = Path("/content/drive/MyDrive/Projeto_EDA_Democracia")
else:
    # Executando localmente a partir da pasta notebooks/.
    PROJECT_ROOT = Path("..").resolve()

# Paths das bases brutas.
PATH_CESOP_SAV = PROJECT_ROOT / "data" / "raw" / "cesop" / "04832.SAV"
PATH_TSE_CSV = (PROJECT_ROOT / "data" / "raw" / "tse"
                / "perfil_comparecimento_abstencao_2022"
                / "perfil_comparecimento_abstencao_2022_BRASIL.csv")

# Pasta de saída dos dados tratados (criada se não existir).
PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Ambiente Colab? ", IN_COLAB)
print("CESOP SAV existe?", PATH_CESOP_SAV.exists())
print("TSE CSV existe? ", PATH_TSE_CSV.exists())

---
## Parte I — CESOP

### 1. Carregamento da base SPSS

A base do CESOP vem em formato **SPSS (`.SAV`)**. Usamos `pyreadstat`, que retorna o
DataFrame **e** um objeto `meta` com os rótulos das colunas e os mapeamentos
código→categoria embutidos no arquivo. **Por quê:** esses metadados são essenciais para
recodificar os valores numéricos em textos legíveis mais adiante, sem depender de uma
tabela externa.

In [ ]:
# Lê o .SAV e separa dados (df) dos metadados (meta).
df_cesop, meta_cesop = pyreadstat.read_sav(str(PATH_CESOP_SAV))
print("Linhas e colunas:", df_cesop.shape)

### 2. Inspeção inicial da estrutura

Antes de qualquer tratamento, observamos os tipos de dado e o preenchimento das colunas.
**Por quê:** identificar de imediato colunas com muitos `NaN` (as perguntas de resposta
múltipla `P2_2`/`P2_3` e `P3_2`..`P3_6`, que só são preenchidas quando o respondente cita
mais de uma opção) e confirmar que, nesta fase, todos os valores ainda são **códigos
numéricos** do SPSS (1.0, 2.0, ...).

In [ ]:
# Tipos de dado e contagem de não-nulos por coluna.
df_cesop.info()

In [ ]:
# Amostra das primeiras linhas — valores ainda em código numérico do SPSS.
df_cesop.head()

### 3. Rótulos e mapeamentos do SPSS

Extraímos dois dicionários dos metadados: (a) o **rótulo descritivo de cada coluna**
(ex.: o texto completo da pergunta) e (b) os **mapeamentos código→categoria** de cada
variável. **Por quê:** o primeiro serve de documentação; o segundo será aplicado para
transformar códigos em rótulos legíveis. Variáveis numéricas livres (`IDADE`, `ID_Ipec`)
não possuem mapeamento e por isso não aparecem na lista de value labels.

In [ ]:
# (a) Rótulo descritivo de cada coluna (dicionário nome -> descrição).
labels_cesop = dict(zip(meta_cesop.column_names, meta_cesop.column_labels))

# (b) Variáveis que possuem mapeamento código -> categoria.
print("Variáveis com value labels:")
print(list(meta_cesop.variable_value_labels.keys()))

### 4. Diagnóstico de nulos e cardinalidade

Verificamos quantos valores ausentes existem por coluna e quantas categorias distintas
cada variável possui. **Por quê:** a cardinalidade orienta as decisões de agrupamento da
etapa de variáveis derivadas (variáveis com muitos níveis, como `ESCOLARIDADE`, serão
agregadas), e a contagem de nulos estabelece uma linha de base para medir o efeito da
limpeza.

In [ ]:
# Valores ausentes por coluna (estado inicial, antes da limpeza).
print("=== NaN por coluna (estado inicial) ===")
print(df_cesop.isnull().sum())

# Cardinalidade (nº de categorias distintas) por coluna.
print("\n=== Cardinalidade por coluna ===")
for coluna in df_cesop.columns:
    print(f"{coluna}: {df_cesop[coluna].nunique()} valores distintos")

### 5. Estruturação: padronização de nomes e datas

Renomeamos as variáveis de pergunta para o padrão do dicionário de dados
(`P1A` → `P_01A`, etc.) e convertemos `DATA_ENTREVISTA` para `datetime`. **Por quê:**
nomes consistentes com a documentação evitam confusão nos cruzamentos e no texto do
trabalho; o tipo `datetime` habilita filtros e ordenações temporais corretas.

In [ ]:
# Renomeia as variáveis de pergunta para o padrão do dicionário (P_01A em vez de P1A).
rename_map = {
    "P1A": "P_01A", "P1B": "P_01B", "P1C": "P_01C",
    "P2_1": "P_02_1", "P2_2": "P_02_2", "P2_3": "P_02_3",
    "P3_1": "P_03_1", "P3_2": "P_03_2", "P3_3": "P_03_3",
    "P3_4": "P_03_4", "P3_5": "P_03_5", "P3_6": "P_03_6",
    "P4":   "P_04",
}
df_cesop = df_cesop.rename(columns=rename_map)

# DATA_ENTREVISTA chega como string ('YYYY-MM-DD'); convertê-la habilita uso temporal.
df_cesop["DATA_ENTREVISTA"] = pd.to_datetime(df_cesop["DATA_ENTREVISTA"], errors="coerce")

print("Colunas renomeadas:", [v for v in rename_map.values() if v in df_cesop.columns])
print("Dtype DATA_ENTREVISTA:", df_cesop["DATA_ENTREVISTA"].dtype)

### 6. Limpeza de valores especiais (códigos de não-resposta)

Códigos especiais do SPSS representam ausência de resposta e **não** podem ser
confundidos com categorias válidas. Tratamos `99` ("Não respondeu / Não sabe") como
`NaN`. **Por quê fazemos isso por coluna, e não globalmente:** em `REND1`/`REND2` o
código `98` significa *"Não tem rendimento pessoal"* — uma categoria **válida**. Manter a
tabela explícita documenta a decisão e protege contra uma generalização equivocada que
apagasse o `98`.

In [ ]:
# Mapeamento explícito dos códigos de não-resposta para NaN, por coluna.
# Manter a tabela explícita (em vez de aplicar "99 -> NaN" global) documenta a decisão
# e evita corromper REND1/REND2, onde 98 = "Não tem rendimento" é categoria VÁLIDA.
codigos_nan_por_coluna = {
    "RELIGIAO": [99],
    "REND1":    [99],
    "REND2":    [99],
    "P_01A":    [99], "P_01B": [99], "P_01C": [99],
    "P_02_1":   [99], "P_02_2": [99], "P_02_3": [99],
    "P_03_1":   [99], "P_03_2": [99], "P_03_3": [99],
    "P_03_4":   [99], "P_03_5": [99], "P_03_6": [99],
    "P_04":     [99],
}

# Conta NaN antes e depois para dar visibilidade ao efeito da substituição.
antes = df_cesop[list(codigos_nan_por_coluna.keys())].isna().sum()
for col, codigos in codigos_nan_por_coluna.items():
    df_cesop[col] = df_cesop[col].replace(codigos, np.nan)
depois = df_cesop[list(codigos_nan_por_coluna.keys())].isna().sum()

pd.DataFrame({"NaN_antes": antes, "NaN_depois": depois, "delta": depois - antes})

### 7. Aplicação dos rótulos legíveis (*value labels*)

Substituímos os códigos numéricos pelos rótulos textuais de cada variável categórica e
convertemos as colunas para o tipo `category`. **Por quê:** rótulos legíveis tornam as
análises e gráficos autoexplicativos; o tipo `category` economiza memória e habilita
ordenação categórica. O `.map()` é aplicado **depois** da limpeza de `99`, preservando os
`NaN` recém-inseridos.

In [ ]:
# Reindexa as chaves do dicionário SPSS para refletir os nomes renomeados (P1A -> P_01A).
value_labels = {}
for col_original, mapa in meta_cesop.variable_value_labels.items():
    col_atual = rename_map.get(col_original, col_original)
    if col_atual in df_cesop.columns:
        value_labels[col_atual] = mapa

# Aplica os rótulos e converte para category. Colunas sem value labels
# (IDADE, ID_Ipec, DATA_ENTREVISTA, TIPO_COLETA) ficam inalteradas.
for col, mapa in value_labels.items():
    df_cesop[col] = df_cesop[col].map(mapa).astype("category")

df_cesop.head()

### 8. Variáveis derivadas: escolaridade e renda agrupadas

Criamos agrupamentos para reduzir a granularidade de variáveis com muitas categorias.
**Por quê:** `ESCOLARIDADE` tem 16 níveis e `REND1`/`REND2` têm faixas finas — agregá-los
torna as tabelas cruzadas e os gráficos legíveis, sem perder o gradiente analítico.

- `ESCOL_GRUPO`: agrega os 16 níveis em **5 grupos ordenados** (Baixa escolaridade →
  Ensino fundamental → Ensino médio → Superior incompleto → Superior completo).
- `RENDA_PESSOAL` / `RENDA_FAMILIAR`: agregam `REND1`/`REND2` em **5 faixas ordenadas**.

> **Correção aplicada na revisão:** a ordem das categorias de `ESCOL_GRUPO` foi ajustada
> para que *Superior incompleto* (código 15) venha **antes** de *Superior completo*
> (código 16), corrigindo a ordenação dos eixos em todos os gráficos por escolaridade.

Ao final, validamos a **cobertura** dos mapeamentos: contamos quantos registros com valor
original não-nulo ficaram sem classificação (o que indicaria um rótulo não previsto).

In [ ]:
# ESCOL_GRUPO: agrega os 16 níveis de escolaridade em 5 grandes grupos,
# usando os labels textuais já aplicados (não os códigos).
escol_analfabeto = {
    "Analfabeto", "Sabe ler/ escrever, mas não cursou escola",
}
escol_fundamental = {
    "Pré-escola (ou 1º ano)", "1ª série (ou 2º ano)", "2ª série (ou 3º ano)",
    "3ª série (ou 4º ano)", "4ª série (ou 5º ano)", "5ª série (ou 6º ano)",
    "6ª série (ou 7º ano)", "7ª série (ou 8º ano)", "8ª série (ou 9º ano)",
}
escol_medio = {"1ª série", "2ª série", "3ª série"}
escol_superior_completo = {"Superior completo"}
escol_superior_incompleto = {"Superior incompleto"}


def _classificar_escol(valor):
    if valor in escol_analfabeto:
        return "Baixa escolaridade"
    if valor in escol_fundamental:
        return "Ensino fundamental"
    if valor in escol_medio:
        return "Ensino médio"
    if valor in escol_superior_incompleto:
        return "Superior incompleto"
    if valor in escol_superior_completo:
        return "Superior completo"
    return np.nan


# Ordem CORRIGIDA: incompleto antes de completo (gradiente educacional crescente).
df_cesop["ESCOL_GRUPO"] = pd.Categorical(
    df_cesop["ESCOLARIDADE"].astype(object).map(_classificar_escol),
    categories=["Baixa escolaridade", "Ensino fundamental", "Ensino médio",
                "Superior incompleto", "Superior completo"],
    ordered=True,
)

# RENDA: combina os extremos para reduzir esparsidade nas tabelas cruzadas.
sem_rendimento = {"NÃO TEM RENDIMENTO PESSOAL"}
renda_ate_1sm = {"ATÉ 1"}
renda_1_5sm = {"MAIS DE 1 A 2", "MAIS DE 2 A 5"}
renda_acima_5sm = {"MAIS DE 5 A 10", "MAIS DE 10 A 20"}
renda_acima_20 = {"MAIS DE 20"}


def _classificar_renda(valor):
    if valor in sem_rendimento:
        return "Sem rendimento"
    if valor in renda_ate_1sm:
        return "Até 1 SM"
    if valor in renda_1_5sm:
        return "1 a 5 SM"
    if valor in renda_acima_5sm:
        return "Acima de 5 SM"
    if valor in renda_acima_20:
        return "Acima de 20 SM"
    return np.nan


ordem_renda = ["Sem rendimento", "Até 1 SM", "1 a 5 SM", "Acima de 5 SM", "Acima de 20 SM"]
df_cesop["RENDA_PESSOAL"] = pd.Categorical(
    df_cesop["REND1"].astype(object).map(_classificar_renda),
    categories=ordem_renda, ordered=True,
)
df_cesop["RENDA_FAMILIAR"] = pd.Categorical(
    df_cesop["REND2"].astype(object).map(_classificar_renda),
    categories=ordem_renda, ordered=True,
)

df_cesop[["ESCOLARIDADE", "ESCOL_GRUPO", "REND1", "RENDA_PESSOAL", "REND2", "RENDA_FAMILIAR"]].head(10)

In [ ]:
# Validação de cobertura dos mapeamentos.
# Um valor ORIGINAL não-nulo que virou NaN no grupo indica um rótulo não previsto
# (ex.: diferença de acentuação/caixa) — uma falha silenciosa que queremos detectar.
def _checar_cobertura(col_origem, col_destino):
    nao_classif = df_cesop[df_cesop[col_origem].notna() & df_cesop[col_destino].isna()]
    n = len(nao_classif)
    print(f"{col_destino}: {n} registro(s) com origem preenchida ficaram sem classificação")
    if n > 0:
        print("  Rótulos não mapeados:", sorted(nao_classif[col_origem].astype(str).unique()))

_checar_cobertura("ESCOLARIDADE", "ESCOL_GRUPO")
_checar_cobertura("REND1", "RENDA_PESSOAL")
_checar_cobertura("REND2", "RENDA_FAMILIAR")

### 9. Distribuições univariadas (verificação de sanidade)

Conferimos a contagem de respondentes em cada variável de perfil. **Por quê:** esta é uma
*checagem de sanidade* do tratamento — confirma que os grupos derivados têm tamanhos
plausíveis, que não há categoria vazia inesperada e que os rótulos foram aplicados. Não é
análise exploratória; é controle de qualidade da preparação.

In [ ]:
# Distribuição de cada variável de perfil (controle de qualidade, não análise).
variaveis_perfil = ["SEXO", "FX_ID", "ESCOL_GRUPO", "RENDA_PESSOAL",
                    "RENDA_FAMILIAR", "REGIAO", "RACA", "RELIGIAO"]
for var in variaveis_perfil:
    if var in df_cesop.columns:
        print(f"=== {var} ===")
        print(df_cesop.groupby(var, observed=True).size())
        print()

### 10. Validação final e salvamento

Conferimos os tipos finais e a completude, então salvamos a base em **Parquet**.
**Por quê Parquet:** preserva os dtypes `category` sem recodificação ao reler e reduz
drasticamente o tamanho em disco. Salvamos também o **dicionário de rótulos das colunas**
em JSON, para servir de referência reproduzível no texto do trabalho.

In [ ]:
# Resumo final de dtypes e completude após o tratamento.
print("=== Dtypes ===")
print(df_cesop.dtypes)
print("\n=== Total de NaN por coluna (top 20) ===")
print(df_cesop.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
# Salva a base CESOP tratada e o dicionário de rótulos das colunas.
import json

path_out_cesop = PATH_PROCESSED / "cesop_clean.parquet"
df_cesop.to_parquet(path_out_cesop, index=False)
print(f"Salvo: {path_out_cesop.name} ({path_out_cesop.stat().st_size / 1024:.1f} KB)")

path_labels = PATH_PROCESSED / "cesop_labels.json"
with open(path_labels, "w", encoding="utf-8") as f:
    json.dump({k: str(v) for k, v in labels_cesop.items()}, f, ensure_ascii=False, indent=2)
print(f"Salvo: {path_labels.name} (dicionário de rótulos das colunas)")

---
## Parte II — TSE

### 11. Carregamento otimizado

O arquivo `perfil_comparecimento_abstencao_2022_BRASIL.csv` tem ~2,3 GB. Carregamos
**apenas as colunas necessárias** para o cruzamento com a CESOP (perfil demográfico +
contagens de comparecimento/abstenção). **Por quê:** ler o arquivo inteiro estouraria a
memória do Colab; `usecols` reduz o consumo a uma fração. O arquivo usa separador `;` e
encoding **`latin1`** (padrão das bases do TSE; caracteres acentuados como *"Não
informado"* só são lidos corretamente com esse encoding).

In [ ]:
# Carregamento parcial do TSE: somente as colunas usadas nos cruzamentos.
colunas_tse = [
    "SG_UF", "NM_MUNICIPIO",
    "DS_GENERO", "DS_ESTADO_CIVIL",
    "DS_FAIXA_ETARIA", "DS_GRAU_ESCOLARIDADE", "DS_COR_RACA",
    "QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCAO",
]

df_tse = pd.read_csv(
    PATH_TSE_CSV,
    sep=";",
    encoding="latin1",   # encoding correto das bases do TSE
    usecols=colunas_tse,
)
print("Linhas e colunas:", df_tse.shape)
df_tse.head()

### 12. Diagnóstico do TSE

Inspecionamos tipos, completude e estatísticas das colunas numéricas. **Por quê:**
confirmar que as contagens (`QT_*`) foram lidas como inteiros, localizar eventuais nulos
e ter uma visão inicial das ordens de grandeza antes de calcular taxas.

In [ ]:
# Tipos e completude.
df_tse.info()

# Valores ausentes por coluna.
print("\n=== NaN por coluna ===")
print(df_tse.isnull().sum())

# Estatísticas das colunas numéricas.
df_tse.describe()

### 13. Limpeza do TSE

A base está na granularidade máxima: cada linha é uma combinação
(UF × município × zona × perfil demográfico). Aplicamos três tratamentos:

1. **Remoção de linhas com `QT_APTOS == 0`** — combinações sem eleitores aptos não
   contribuem para taxas (dividiriam por zero) e só poluem as agregações.
2. **Padronização de strings** (remoção de espaços extras; `SG_UF` em maiúsculas) —
   defesa contra variações de origem que quebrariam *joins*.
3. **Conversão das textuais para `category`** — redução de memória relevante dado o
   tamanho da base.

In [ ]:
# 1) Remove combinações sem eleitores aptos.
linhas_antes = len(df_tse)
df_tse = df_tse[df_tse["QT_APTOS"] > 0].copy()
print(f"Linhas removidas (QT_APTOS=0): {linhas_antes - len(df_tse):,}")
print(f"Linhas restantes: {len(df_tse):,}")

# 2) Normaliza espaços nas colunas textuais e padroniza UF em maiúsculas.
colunas_texto = ["SG_UF", "NM_MUNICIPIO", "DS_GENERO", "DS_ESTADO_CIVIL",
                 "DS_FAIXA_ETARIA", "DS_GRAU_ESCOLARIDADE", "DS_COR_RACA"]
for col in colunas_texto:
    df_tse[col] = df_tse[col].astype(str).str.strip()
df_tse["SG_UF"] = df_tse["SG_UF"].str.upper()

# 3) Converte textuais para category (economia de memória).
for col in colunas_texto:
    df_tse[col] = df_tse[col].astype("category")

df_tse.dtypes

### 14. Variáveis derivadas: taxas e região

Criamos as métricas centrais do cruzamento com a CESOP:

- `TAXA_COMPARECIMENTO` = `QT_COMPARECIMENTO / QT_APTOS`
- `TAXA_ABSTENCAO` = `QT_ABSTENCAO / QT_APTOS`
- `REGIAO`: derivada da UF, replicando a categorização IBGE para viabilizar o cruzamento
  geográfico com a CESOP.

**Por quê as taxas como proporção [0, 1]:** mantê-las normalizadas facilita as validações
de sanidade adiante; a conversão para percentual fica a cargo da etapa de visualização.

In [ ]:
# Taxas (proporção em [0, 1]).
df_tse["TAXA_COMPARECIMENTO"] = df_tse["QT_COMPARECIMENTO"] / df_tse["QT_APTOS"]
df_tse["TAXA_ABSTENCAO"] = df_tse["QT_ABSTENCAO"] / df_tse["QT_APTOS"]

# REGIAO derivada da UF (categorização IBGE). ZZ = voto no exterior.
uf_para_regiao = {
    "AC": "Norte", "AM": "Norte", "AP": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MS": "Centro-Oeste", "MT": "Centro-Oeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
    "ZZ": "Exterior",
}
df_tse["REGIAO"] = df_tse["SG_UF"].astype(str).map(uf_para_regiao).astype("category")

df_tse[["SG_UF", "REGIAO", "TAXA_COMPARECIMENTO", "TAXA_ABSTENCAO"]].head()

### 15. Distribuições do TSE (verificação)

Conferimos as contagens de combinações por escolaridade, região e gênero. **Por quê:**
mesma lógica de controle de qualidade do CESOP — garante que os rótulos foram lidos
corretamente (encoding) e que a derivação de `REGIAO` cobriu todas as UFs.

In [ ]:
# Contagem de combinações por variável-chave (controle de qualidade).
for var in ["DS_GRAU_ESCOLARIDADE", "REGIAO", "DS_GENERO", "DS_COR_RACA"]:
    print(f"=== {var} ===")
    print(df_tse.groupby(var, observed=True).size())
    print()

### 16. Agregações: por UF e por perfil demográfico

Geramos dois recortes usados nas análises:

- `df_tse_uf`: totais e taxas **por UF** — permite ranquear estados.
- `df_tse_perfil`: totais e taxas **por perfil demográfico nacional**
  (gênero × faixa etária × escolaridade × raça) — base de comparação com a amostra CESOP.

> **Nota metodológica importante:** as taxas agregadas são calculadas a partir das
> **somas** das contagens, e **não** como média das taxas-linha. Isso garante uma média
> *ponderada* pelo número de eleitores — zonas pequenas não distorcem o indicador.

In [ ]:
# Agregação por UF: soma dos contingentes e taxa global ponderada.
df_tse_uf = (
    df_tse
    .groupby("SG_UF", observed=True)
    .agg(
        QT_APTOS=("QT_APTOS", "sum"),
        QT_COMPARECIMENTO=("QT_COMPARECIMENTO", "sum"),
        QT_ABSTENCAO=("QT_ABSTENCAO", "sum"),
    )
    .reset_index()
)
df_tse_uf["TAXA_COMPARECIMENTO"] = df_tse_uf["QT_COMPARECIMENTO"] / df_tse_uf["QT_APTOS"]
df_tse_uf["TAXA_ABSTENCAO"] = df_tse_uf["QT_ABSTENCAO"] / df_tse_uf["QT_APTOS"]
df_tse_uf["REGIAO"] = df_tse_uf["SG_UF"].astype(str).map(uf_para_regiao)
df_tse_uf.head()

In [ ]:
# Agregação por perfil demográfico nacional (mesmo grão para comparar com a CESOP).
df_tse_perfil = (
    df_tse
    .groupby(
        ["DS_GENERO", "DS_FAIXA_ETARIA", "DS_GRAU_ESCOLARIDADE", "DS_COR_RACA"],
        observed=True,
    )
    .agg(
        QT_APTOS=("QT_APTOS", "sum"),
        QT_COMPARECIMENTO=("QT_COMPARECIMENTO", "sum"),
        QT_ABSTENCAO=("QT_ABSTENCAO", "sum"),
    )
    .reset_index()
)
df_tse_perfil["TAXA_COMPARECIMENTO"] = df_tse_perfil["QT_COMPARECIMENTO"] / df_tse_perfil["QT_APTOS"]
df_tse_perfil["TAXA_ABSTENCAO"] = df_tse_perfil["QT_ABSTENCAO"] / df_tse_perfil["QT_APTOS"]
df_tse_perfil.head()

### 17. Validação e salvamento

Validamos por `assert` que todas as taxas estão no intervalo `[0, 1]`. **Por quê:** uma
taxa fora desse intervalo denunciaria erro de cálculo ou inconsistência da fonte — melhor
falhar aqui, de forma explícita, do que propagar dados inválidos. Em seguida salvamos os
três recortes em Parquet.

In [ ]:
# Sanidade: todas as taxas devem ficar em [0, 1].
assert df_tse["TAXA_COMPARECIMENTO"].between(0, 1).all(), "Taxa de comparecimento fora de [0, 1]"
assert df_tse["TAXA_ABSTENCAO"].between(0, 1).all(), "Taxa de abstenção fora de [0, 1]"

print("=== Shapes finais ===")
print(f"df_tse (linhas-detalhe) : {df_tse.shape}")
print(f"df_tse_uf (por UF)      : {df_tse_uf.shape}")
print(f"df_tse_perfil (perfil)  : {df_tse_perfil.shape}")

In [ ]:
# Salva os três recortes do TSE em Parquet.
path_tse_clean = PATH_PROCESSED / "tse_clean.parquet"
path_tse_uf = PATH_PROCESSED / "tse_uf.parquet"
path_tse_perfil = PATH_PROCESSED / "tse_perfil.parquet"

df_tse.to_parquet(path_tse_clean, index=False)
df_tse_uf.to_parquet(path_tse_uf, index=False)
df_tse_perfil.to_parquet(path_tse_perfil, index=False)

for p in (path_tse_clean, path_tse_uf, path_tse_perfil):
    print(f"Salvo: {p.name} ({p.stat().st_size / (1024**2):.2f} MB)")

---
## 18. Resumo do pipeline

Bases preparadas e salvas em `data/processed/`:

| Arquivo | Origem | Conteúdo |
|---|---|---|
| `cesop_clean.parquet` | CESOP `04832.SAV` | Respondentes com perguntas recodificadas, variáveis derivadas (`ESCOL_GRUPO`, `RENDA_PESSOAL`, `RENDA_FAMILIAR`) e dtypes ajustados |
| `cesop_labels.json` | CESOP `04832.SAV` | Dicionário de rótulos descritivos das colunas |
| `tse_clean.parquet` | TSE BRASIL 2022 | Linhas-detalhe (município × zona × perfil) com taxas calculadas |
| `tse_uf.parquet` | Agregado | Totais e taxas por UF |
| `tse_perfil.parquet` | Agregado | Totais e taxas por perfil demográfico nacional |

**Próximo passo:** análises exploratórias e cruzamento CESOP × TSE no notebook
`02_analises_exploratorias_parte_a.ipynb`.